In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
cd /content/drive/MyDrive/MS/rag_code_audit/

/content/drive/MyDrive/MS/rag_code_audit


In [4]:
pip install -q langgraph openai chromadb pydantic sentence-transformers pandas scikit-learn pypdf

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 66.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 83.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 59.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/6

# Storing pdf into Vector DB



In [14]:
import pypdf
import chromadb

chroma_client = chromadb.Client()
collection = chroma_client.get_or_create_collection(name="nist_ai_rmf_full")


pdf_path = "pdf/fairlending.pdf"
reader = pypdf.PdfReader(pdf_path)

full_text = ""
print(f"Reading {len(reader.pages)} pages from NIST AI RMF PDF...")
for page in reader.pages:
    text = page.extract_text()
    if text:
        full_text += text + " "

raw_paragraphs = full_text.split(".\n")

Reading 49 pages from NIST AI RMF PDF...


In [19]:
print(full_text)

 
 
 
__________________________________________________________________ 
 
 
__________________________________________________________________ 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
Office of the Comptroller of the Currency 
Federal Deposit Insurance Corporation 
Federal Reserve Board 
Office of Thrift Supervision 
National Credit Union Administration 
INTERAGENCY FAIR LENDING 
EXAMINATION PROCEDURES 
          A u g u s t 2 0 0 9   
 
 
 
 
 
                  
 
           
               
       
   
     
  
    
   
     
         
 
        
 
         
          
    
  
 
               
     
    
   
            
          
 
     
 
 
 
 
  
 
  
 
 
  
CONTENTS 
INTRODUCTION i
 
PART I - EXAMINATION SCOPE GUIDELINES 1 
Background 1 
Step One – Develop an Overview 5 
Step Two - Identify Compliance Program Discrimination Risk Factors 6 
Step Three - Review Residential Loan Products 7 
Step Four - Identify Residential Lending Discrimination Risk Factors 8 
St

In [20]:
print(raw_paragraphs)

[' \n \n \n__________________________________________________________________ \n \n \n__________________________________________________________________ \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \nOffice of the Comptroller of the Currency \nFederal Deposit Insurance Corporation \nFederal Reserve Board \nOffice of Thrift Supervision \nNational Credit Union Administration \nINTERAGENCY FAIR LENDING \nEXAMINATION PROCEDURES \n          A u g u s t 2 0 0 9   \n \n \n \n \n \n                  \n \n           \n               \n       \n   \n     \n  \n    \n   \n     \n         \n \n        \n \n         \n          \n    \n  \n \n               \n     \n    \n   \n            \n          \n \n     \n \n \n \n \n  \n \n  \n \n \n  \nCONTENTS \nINTRODUCTION i\n \nPART I - EXAMINATION SCOPE GUIDELINES 1 \nBackground 1 \nStep One – Develop an Overview 5 \nStep Two - Identify Compliance Program Discrimination Risk Factors 6 \nStep Three - Review Residenti

In [6]:
documents = []
ids = []

for i, raw_p in enumerate(raw_paragraphs):
    clean_paragraph = " ".join(raw_p.split())

    if len(clean_paragraph) > 50:
        documents.append(clean_paragraph)
        ids.append(f"paragraph_{len(documents)}")

collection.add(
    documents=documents,
    ids=ids
)
print(f"Successfully stored {len(documents)} semantic paragraph chunks from the NIST PDF into ChromaDB!")

/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:01<00:00, 56.2MiB/s]


Successfully stored 288 semantic paragraph chunks from the NIST PDF into ChromaDB!


In [7]:
results = collection.query(query_texts=["What are the challenges of AI risk management"], n_results=5)
results

{'ids': [['paragraph_25',
   'paragraph_39',
   'paragraph_21',
   'paragraph_17',
   'paragraph_23']],
 'embeddings': None,
 'documents': [['1.2 Challenges for AI Risk Management Several challenges are described below. They should be taken into account when managing risks in pursuit of AI trustworthiness',
   'To the extent that challenges for specifying AI risk tolerances remain unresolved, there may be contexts where a risk management framework is not yet readily applicable for mitigating negative AI risks',
   'While risk management processes generally address negative impacts, this Framework of- fers approaches to minimize anticipated negative impacts of AI systems and identify op- portunities to maximize positive impacts. Effectively managing the risk of potential harms could lead to more trustworthy AI systems and unleash potential benefits to people (individ- uals, communities, and society), organizations, and systems/ecosystems. Risk management can enable AI developers and use

In [ ]:
results = collection.query(query_texts=["extract all the policies and conditions required to be checked for model development"], n_results=10)
results1 = collection.query(query_texts=["extract all the policies and conditions required to be checked for the model development code"], n_results=10)
flattened_rules = "\n".join(results['documents'][0])

In [11]:
chroma_client = chromadb.PersistentClient(path="./chroma_db")
collection = chroma_client.get_or_create_collection(name="nist_ai_rmf_full")


pdf_path = "nist_ai_rmf.pdf"
reader = pypdf.PdfReader(pdf_path)

full_text = ""
print(f"Reading {len(reader.pages)} pages from NIST AI RMF PDF...")
for page in reader.pages:
    text = page.extract_text()
    if text:
        full_text += text + " "

raw_paragraphs = full_text.split(".\n")

documents = []
ids = []

for i, raw_p in enumerate(raw_paragraphs):
    clean_paragraph = " ".join(raw_p.split())

    if len(clean_paragraph) > 50:
        documents.append(clean_paragraph)
        ids.append(f"paragraph_{len(documents)}")

collection.add(
    documents=documents,
    ids=ids
)
print(f"Successfully stored {len(documents)} semantic paragraph chunks from the NIST PDF into ChromaDB!")

Reading 48 pages from NIST AI RMF PDF...
Successfully stored 288 semantic paragraph chunks from the NIST PDF into ChromaDB!


# Policies Retrieval

In [8]:
import sys, chromadb
from typing import TypedDict, List, Dict, Any
from openai import OpenAI
from langgraph.graph import StateGraph, END

# Initialize the OpenAI client pointing to your Mac's ngrok URL
client = OpenAI(
    base_url="https://backup-kerchief-aloof.ngrok-free.dev/v1", # The /v1 is required for Ollama's OpenAI compatibility
    api_key="ollama" # Required by the SDK, but ignored by Ollama
)

# Set your target local model (must match the model you pulled in Ollama)
TARGET_MODEL = "qwen2.5:7b-instruct"

# Legal Extractor Node
def policy(query_txt) -> Dict[str, Any]:
    results = collection.query(query_texts=[query_txt], n_results=5)
    flattened_rules = "\n".join(results['documents'][0])

    prompt = f"Policies:\n{flattened_rules}\nExtract policy rules and math condition if any."
    response = client.chat.completions.create(
        model=TARGET_MODEL,
        messages=[{"role": "user", "content": prompt}]
    )

    return response.choices[0].message.content


In [9]:
response = policy("What are the challenges of AI risk management")

'### Policy Rules Extracted\n\n1. **Challenges for Specifying AI Risk Tolerances**:\n   - If challenges in specifying AI risk tolerances remain unresolved, a risk management framework may not be applicable or readily usable.\n   \n2. **Risk Management Framework**:\n   - The framework offers approaches to minimize anticipated negative impacts of AI systems and identify opportunities to maximize positive impacts.\n   - Effective risk management processes generally address negative impacts; the framework can help mitigate these impacts.\n\n3. **Beneficial Outcomes from Risk Management**:\n   - Effectively managing potential harms of AI systems could lead to more trustworthy AI systems, benefiting individuals, communities, and society.\n   - Risk management enables understanding of impact, accounting for model limitations, and improving overall system performance and trustworthiness.\n   - This can increase the likelihood that AI technologies will be used in beneficial ways.\n\n4. **Framin

In [10]:
print(response)

### Policy Rules Extracted

1. **Challenges for Specifying AI Risk Tolerances**:
   - If challenges in specifying AI risk tolerances remain unresolved, a risk management framework may not be applicable or readily usable.
   
2. **Risk Management Framework**:
   - The framework offers approaches to minimize anticipated negative impacts of AI systems and identify opportunities to maximize positive impacts.
   - Effective risk management processes generally address negative impacts; the framework can help mitigate these impacts.

3. **Beneficial Outcomes from Risk Management**:
   - Effectively managing potential harms of AI systems could lead to more trustworthy AI systems, benefiting individuals, communities, and society.
   - Risk management enables understanding of impact, accounting for model limitations, and improving overall system performance and trustworthiness.
   - This can increase the likelihood that AI technologies will be used in beneficial ways.

4. **Framing Risk**:
   - 

In [22]:
pip install "unstructured[pdf]"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 13.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.1/69.1 kB 3.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.3/80.3 kB 4.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.3/62.3 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.9/109.9 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 22.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.2/542.2 kB 20.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 453.8/453.8 kB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 36.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 29.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 40.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.6/155.6 kB 9.1 MB/s eta 0:00:00
   ━

In [1]:
from unstructured.partition.pdf import partition_pdf
from unstructured.chunking.title import chunk_by_title

In [34]:
import chromadb

In [35]:
!pwd

/content/drive/MyDrive/MS/rag_code_audit


In [36]:
pdf_path="pdf/fairlending.pdf"
db_path="./chroma_db"
collection_name="nist_ai_rmf_unstructured"

print(f"Parsing {pdf_path}...")

# 1. Partition the PDF into structured elements
# strategy="hi_res" uses layout detection models to properly identify
# columns, tables, headers, and paragraphs.
elements = partition_pdf(
    filename=pdf_path,
    strategy="hi_res"
)

print(f"Extracted {len(elements)} elements. Chunking...")

# 2. Chunk the elements semantically
# chunk_by_title groups text blocks under their respective section headers.
chunks = chunk_by_title(
    elements,
    max_characters=1000,
    new_after_n_chars=800,
    combine_text_under_n_chars=200
)

print(f"Created {len(chunks)} semantic chunks. Loading into ChromaDB...")


Parsing pdf/fairlending.pdf...


Extracted 608 elements. Chunking...
Created 178 semantic chunks. Loading into ChromaDB...


In [43]:
db_path = "./chroma_db/fairlending"
collection_name="fairlending_unstructured"
# 3. Initialize ChromaDB Persistent Client
# chroma_client = chromadb.PersistentClient(path=db_path)
chroma_client = chromadb.PersistentClient(db_path)
collection = chroma_client.get_or_create_collection(name=collection_name)

# 4. Prepare data for database insertion
documents = []
metadatas = []
ids = []

for i, chunk in enumerate(chunks):
    documents.append(chunk.text)

    # Unstructured automatically captures rich metadata (page numbers, source, etc.)
    metadata = chunk.metadata.to_dict()

    # ChromaDB requires metadata values to be strict strings, ints, or floats.
    # This cleans out complex objects or null values.
    clean_metadata = {k: str(v) for k, v in metadata.items() if v is not None}
    metadatas.append(clean_metadata)

    ids.append(f"{pdf_path}_chunk_{i}")

# 5. Insert into ChromaDB
# Chroma handles the embedding automatically using its default model
# (or you can pass your own embedding function here).
collection.add(
    documents=documents,
    metadatas=metadatas,
    ids=ids
)

print(f"Pipeline complete! {len(documents)} chunks successfully dumped into the vector database.")


Pipeline complete! 178 chunks successfully dumped into the vector database.


In [54]:
print(chunks[10])
# print(elements[4])
# results = collection.query(query_texts=[query_txt], n_results=5)

Finally, the FHAct requires lenders to make reasonable accommodations for a person with disabilities when such accommodations are necessary to afford the person an equal opportunity to apply for credit.


# Code Audit

In [ ]:
with open('my_test_file.py', 'w') as file:
  file.write("""
  import pandas as pd
  import numpy as np

  def train_and_predict():
      np.random.seed(42)
      n_samples = 1000
      gender = np.random.binomial(1, 0.5, n_samples)
      housing_status = np.array([np.random.binomial(1, 0.8 if g==1 else 0.3) for g in gender])
      income = np.random.normal(50000, 15000, n_samples)
      credit_score = income * 0.01 + housing_status * 100 + np.random.normal(0, 50, n_samples)

      df = pd.DataFrame({'gender': gender, 'housing_status': housing_status, 'income': income, 'credit_score': credit_score})
      features = df[['housing_status', 'income', 'credit_score']]
      df['approved'] = (features['credit_score'] > 500).astype(int)
      return df""")

print("File created successfully!")

In [ ]:
import sys, chromadb
from typing import TypedDict, List, Dict, Any
from openai import OpenAI
from langgraph.graph import StateGraph, END

# Initialize the OpenAI client pointing to your Mac's ngrok URL
client = OpenAI(
    base_url="https://backup-kerchief-aloof.ngrok-free.dev/v1", # The /v1 is required for Ollama's OpenAI compatibility
    api_key="ollama" # Required by the SDK, but ignored by Ollama
)

# Set your target local model (must match the model you pulled in Ollama)
TARGET_MODEL = "qwen2.5:7b-instruct"

# Define Graph State
class AuditorState(TypedDict):
    source_code: str
    extracted_criteria: List[str]
    audit_findings: str
    sandbox_metrics: Dict[str, Any]
    critic_feedback: str
    compliance_status: str
    iteration_count: int

# Legal Extractor Node
def legal_extractor_node(state: AuditorState) -> Dict[str, Any]:
    results = collection.query(query_texts=["fairness bias proxy protected impact ratio"], n_results=3)
    flattened_rules = "\n".join(results['documents'][0])

    prompt = f"Review frameworks:\n{flattened_rules}\nExtract math and qualitative conditions."
    response = client.chat.completions.create(
        model=TARGET_MODEL,
        messages=[{"role": "user", "content": prompt}]
    )

    return {"extracted_criteria": [response.choices[0].message.content], "iteration_count": state.get("iteration_count", 0) + 1}

# Code Auditor Node
def code_auditor_node(state: AuditorState) -> Dict[str, Any]:
    prompt = f"Target Script:\n{state['source_code']}\nCriteria:\n{state['extracted_criteria'][0]}\nIdentify proxy variables."
    response = client.chat.completions.create(
        model=TARGET_MODEL,
        messages=[{"role": "user", "content": prompt}]
    )

    return {"audit_findings": response.choices[0].message.content}

In [ ]:
# Sandbox Execution Node
def sandbox_execution_node(state: AuditorState) -> Dict[str, Any]:
    try:
        # Dynamically reload the target module so changes made by remediation are caught fresh
        if 'target_model' in sys.modules: del sys.modules['target_model']
        import target_model
        results_df = target_model.train_and_predict()

        # Check impact across housing_status
        p_priv = results_df[results_df['housing_status'] == 1]['approved'].mean()
        p_unpriv = results_df[results_df['housing_status'] == 0]['approved'].mean()
        di_ratio = float(p_unpriv / p_priv) if p_priv > 0 else 0.0
        return {"sandbox_metrics": {"disparate_impact_ratio": di_ratio}}
    except Exception as e:
        return {"sandbox_metrics": {"error": str(e), "disparate_impact_ratio": 0.0}}

# Verifier Node
def verifier_node(state: AuditorState) -> Dict[str, Any]:
    di_ratio = state['sandbox_metrics'].get('disparate_impact_ratio', 1.0)
    # Enforce standard legal constraint (0.80)
    if di_ratio < 0.80:
        return {"compliance_status": "REVISION_NEEDED", "critic_feedback": f"CRITICAL FAIL: DI Ratio {di_ratio:.3f}"}
    return {"compliance_status": "APPROVED", "critic_feedback": "PASS: Disparate Impact Ratio is compliant."}

# Remediation Node
def remediation_node(state: AuditorState) -> Dict[str, Any]:
    prompt = f"Code:\n{state['source_code']}\nFeedback:\n{state['critic_feedback']}\nRewrite train_and_predict() to clear bias by dropping proxy."
    response = client.chat.completions.create(
        model=TARGET_MODEL,
        messages=[{"role": "user", "content": prompt}]
    )

    # Strip down formatting to get clean code
    clean_code = response.choices[0].message.content.replace("```python", "").replace("```", "").strip()
    with open("my_test_file.py", "w") as f:
        f.write(clean_code)
    return {
        "source_code": clean_code,
        "iteration_count": state.get('iteration_count', 0) + 1
        }

# Routing Logic
def determine_routing(state: AuditorState):
    if state['iteration_count'] >= 3: return END
    return "remediate" if state['compliance_status'] == "REVISION_NEEDED" else END

# Compile Graph
builder = StateGraph(AuditorState)
builder.add_node("extract_legal", legal_extractor_node)
builder.add_node("audit_code", code_auditor_node)
builder.add_node("execute_sandbox", sandbox_execution_node)
builder.add_node("verify", verifier_node)
builder.add_node("remediate", remediation_node)

# Define edge pathways
builder.set_entry_point("extract_legal")
builder.add_edge("extract_legal", "audit_code")
builder.add_edge("audit_code", "execute_sandbox")
builder.add_edge("execute_sandbox", "verify")
builder.add_conditional_edges("verify", determine_routing, {"remediate": "remediate", END: END})
builder.add_edge("remediate", "audit_code")

compliance_engine = builder.compile()

# Execution Trigger
if __name__ == "__main__":
    with open("my_test_file.py", "r") as f:
        initial_code = f.read()
    final_output = compliance_engine.invoke({"source_code": initial_code, "iteration_count": 0})
    print("\n==============================================")
    print(f" FINAL RESULT: {final_output['compliance_status']}")
    print("==============================================")
    print(final_output['critic_feedback'])


 FINAL RESULT: REVISION_NEEDED
CRITICAL FAIL: DI Ratio 0.000
